# Exploratory Data Analysis

# Data

Half-hourly shortwave solar radiation (GHI, in W/m²) for Skopje (42.00° N, 21.43° E), covering 1990 up to the latest available data.

The data is obtained from Meteosat satellite observations and comes from the EUMETSAT CM SAF SARAH-3 dataset, downloaded through the Open-Meteo Satellite Radiation API.
The values are satellite estimates, not ground measurements.

The dataset contains three types of radiation:
  1. **Direct**: solar radiation on a horizontal surface that comes straight from the sun.
  2. **Diffuse**: refracted and scattered component of solar radiation reaching the surface
  3. **Shortwave (GHI)**: total incoming solar radiation at the surface. $\text{GHI} = \text{Direct} + \text{Diffuse}$

All values are averages from the previous 30 minutes, and times are in UTC.

In [171]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import pandas as pd
import plotly.express as px

from solar_predictor.data import drop_incomplete_tail, read_sarah3
from solar_predictor.storage import get_location_prefix
from solar_predictor.utils import LOCATIONS, MONTHS, SEASONS

px.defaults.height = 700

# Read Data

In [173]:
raw_df = read_sarah3(
    get_location_prefix(LOCATIONS["SKOPJE"]["LAT"], LOCATIONS["SKOPJE"]["LONG"])
)

In [174]:
df = raw_df.copy()

df.head()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
datetime,,,,,
1990-01-01 00:00:00+00:00,631152000,0.0,0.0,0.0,0.0
1990-01-01 00:30:00+00:00,631153800,0.0,0.0,0.0,0.0
1990-01-01 01:00:00+00:00,631155600,0.0,0.0,0.0,0.0
1990-01-01 01:30:00+00:00,631157400,0.0,0.0,0.0,0.0
1990-01-01 02:00:00+00:00,631159200,0.0,0.0,0.0,0.0


# Data Quality

## Basic Checks

In [175]:
df.describe()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
count,6.456480e+05,641974.000000,641962.000000,641962.000000,641962.000000
mean,1.212234e+09,175.828710,107.919140,68.331826,185.687606
std,3.354888e+08,266.189507,202.574334,94.506365,294.126137
min,6.311520e+08,0.000000,0.000000,0.000000,0.000000
25%,9.216932e+08,0.000000,0.000000,0.000000,0.000000
50%,1.212234e+09,4.000000,0.000000,2.000000,0.000000
75%,1.502775e+09,279.000000,106.000000,127.000000,320.900000
max,1.793317e+09,1044.000000,925.000000,483.000000,1029.500000


There are no negative values, the maximum values are within the expected range, and some missing values still need to be handled.

## Missing and Duplicated Timestamps

In [176]:
expected = pd.date_range(df.index.min(), df.index.max(), freq="30min")
missing = expected.difference(df.index)

print(f"Expected: {len(expected)}, actual: {len(df)}, missing: {len(missing)}")
print(f"Duplicated timestamps: {df.index.duplicated().sum()}")

Expected: 645648, actual: 645648, missing: 0
Duplicated timestamps: 0


There are no missing or duplicated timestamps.

## Missing Values
While all timestamps are present, they may contain missing values in them.

In [177]:
1 - df.count() / df.shape[0]

timestamp                   0.000000
shortwave_radiation         0.005690
direct_radiation            0.005709
diffuse_radiation           0.005709
direct_normal_irradiance    0.005709
dtype: float64

There are missing values withing the rows with valid timestamps.

### Count per Year

In [178]:
missing_count = df.isna().groupby(df.index.year.rename("year")).sum().reset_index()
px.line(
    missing_count,
    x="year",
    y="shortwave_radiation",
    title="Number of missing periods per year",
)

The average year has about 100 missing values, except three years: 1990 with 400, 2009 with 380 and 2026 with about 530.
The missigness in 1990 and 2009 is probably of technical reasons while the data in 2026 may be still processing.

### Missing Values in the Tail
The satellite needs time to send the data to planet Earth, remove the last rows since they don't contain data.

In [179]:
df.tail()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
datetime,,,,,
2026-10-29 21:30:00+00:00,1793309400,NaN,NaN,NaN,NaN
2026-10-29 22:00:00+00:00,1793311200,NaN,NaN,NaN,NaN
2026-10-29 22:30:00+00:00,1793313000,NaN,NaN,NaN,NaN
2026-10-29 23:00:00+00:00,1793314800,NaN,NaN,NaN,NaN
2026-10-29 23:30:00+00:00,1793316600,NaN,NaN,NaN,NaN


#### Filter

In [180]:
filtered_df = drop_incomplete_tail(df)

print(f"Filtering {len(df) - len(filtered_df)} items.")
df = filtered_df

df.tail()

Filtering 96 items.


,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
datetime,,,,,
2026-10-27 21:30:00+00:00,1793136600,0.0,0.0,0.0,0.0
2026-10-27 22:00:00+00:00,1793138400,0.0,0.0,0.0,0.0
2026-10-27 22:30:00+00:00,1793140200,0.0,0.0,0.0,0.0
2026-10-27 23:00:00+00:00,1793142000,0.0,0.0,0.0,0.0
2026-10-27 23:30:00+00:00,1793143800,0.0,0.0,0.0,0.0


### Example - Period with Missing Values

In [181]:
df.head()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance
datetime,,,,,
1990-01-01 00:00:00+00:00,631152000,0.0,0.0,0.0,0.0
1990-01-01 00:30:00+00:00,631153800,0.0,0.0,0.0,0.0
1990-01-01 01:00:00+00:00,631155600,0.0,0.0,0.0,0.0
1990-01-01 01:30:00+00:00,631157400,0.0,0.0,0.0,0.0
1990-01-01 02:00:00+00:00,631159200,0.0,0.0,0.0,0.0


In [182]:
df_melted = df.reset_index().melt(id_vars=["timestamp", "datetime"])
df_melted.head()

,timestamp,datetime,variable,value
0,631152000,1990-01-01 00:00:00+00:00,shortwave_radiation,0.0
1,631153800,1990-01-01 00:30:00+00:00,shortwave_radiation,0.0
2,631155600,1990-01-01 01:00:00+00:00,shortwave_radiation,0.0
3,631157400,1990-01-01 01:30:00+00:00,shortwave_radiation,0.0
4,631159200,1990-01-01 02:00:00+00:00,shortwave_radiation,0.0


In [183]:
missing_sample = df.loc["1990-01-25":"1990-02-01"]
px.line(
    missing_sample,
    x=missing_sample.index,
    y="shortwave_radiation",
    title="Example of period with missing data",
)

### Count by Time of the Day

In [184]:
fig = px.bar(
    df[df["shortwave_radiation"].isna()]
    .index.to_series()
    .dt.time.value_counts()
    .sort_index(),
    labels={"value": "Count", "datetime": "Time of the Day"},
    title="Count of Missing Values by Time of the Day",
)
fig.update_layout(showlegend=False)
fig.show()

Most of the missing values are during the midnight period (23:30 - 01:30).

### Missing Ranges

In [185]:
missing_rows = df.loc[df.isna().any(axis=1)].reset_index()
range_id = (missing_rows["datetime"].diff() != pd.Timedelta("30min")).cumsum()

ranges = missing_rows.groupby(range_id).agg(
    start_ts=("timestamp", "first"),
    end_ts=("timestamp", "last"),
    start=("datetime", "first"),
    end=("datetime", "last"),
    steps=("datetime", "size"),
)

ranges["hours"] = ranges["steps"] / 2

ranges = ranges.reset_index(drop=True)
ranges

,start_ts,end_ts,start,end,steps,hours
0,631274400,631303200,1990-01-02 10:00:00+00:00,1990-01-02 18:00:00+00:00,17,8.5
1,633463200,633506400,1990-01-27 18:00:00+00:00,1990-01-28 06:00:00+00:00,25,12.5
2,633546000,633546000,1990-01-28 17:00:00+00:00,1990-01-28 17:00:00+00:00,1,0.5
3,633549600,633592800,1990-01-28 18:00:00+00:00,1990-01-29 06:00:00+00:00,25,12.5
4,633632400,633632400,1990-01-29 17:00:00+00:00,1990-01-29 17:00:00+00:00,1,0.5
...,...,...,...,...,...,...
840,1773532800,1773876600,2026-03-15 00:00:00+00:00,2026-03-18 23:30:00+00:00,192,96.0
841,1774915200,1774999800,2026-03-31 00:00:00+00:00,2026-03-31 23:30:00+00:00,48,24.0
842,1782864000,1782948600,2026-07-01 00:00:00+00:00,2026-07-01 23:30:00+00:00,48,24.0
843,1784505600,1784590200,2026-07-20 00:00:00+00:00,2026-07-20 23:30:00+00:00,48,24.0


#### Duration

In [186]:
pd.Series(
    {h: (ranges["hours"] <= h).mean() * 100 for h in [1, 2, 4, 8, 24, 100]}
).round(2)

1       71.36
2       92.07
4       93.96
8       95.03
24      98.70
100    100.00
dtype: float64

Most gaps (90%) last 2 hours or less, but about 5% run longer than 8 hours, likely due to major satellite outages.

## Conclusion

Most missing values occur at night, when there's no sun, so they could be filled with zeros.
Since they make up less than 0.5% of the data, no imputation is applied for now and rows with missing values are dropped for the ML model.
If a future algorithm requires complete data, we'll add imputation then.

# Radiation Patterns

## Shortwave Radiation in the Preceding Hour

In [187]:
fig = px.line(
    df.iloc[-20_000:],
    y="shortwave_radiation",
    labels={
        "shortwave_radiation": "Shortwave Radiation (W/m²)",
        "datetime": "Date and Time",
    },
    title="Shortwave Radiation in the Preceding Hour",
)
fig.update_layout(
    xaxis={
        "rangeslider_visible": True,
        "range": [df.index.max() - pd.DateOffset(days=15), df.index.max()],
    },
)
fig

## Daily Mean Shortwave Radiation

In [188]:
daily = df.resample("D")["shortwave_radiation"].mean().reset_index()

fig = px.line(
    daily,
    x="datetime",
    y="shortwave_radiation",
    labels={"shortwave_radiation": "Shortwave Radiation (W/m²)", "datetime": "Date"},
    title="Daily Mean Shortwave Radiation over Time",
    height=700,
)
end = daily["datetime"].max()
fig.update_layout(
    xaxis={
        "rangeslider_visible": True,
        "range": [end - pd.DateOffset(years=5), end],
    },
)
fig

## Daily Mean Shortwave Radiation by Months

In [189]:
daily_month = daily.assign(month=daily["datetime"].dt.month)

fig = px.box(
    daily_month,
    x="month",
    y="shortwave_radiation",
    labels={"shortwave_radiation": "Shortwave Radiation (W/m²)", "month": "Month"},
    title="Daily Mean Shortwave Radiation by Month",
)
fig.update_xaxes(
    type="category",
    tickvals=list(range(1, 13)),
    ticktext=MONTHS,
)
fig

## Distribution of Shortwave Radiation by Time of the Day in Each Month

In [190]:
fig = px.density_heatmap(
    df.assign(month=df.index.month, time=df.index.strftime("%H-%M")),
    x="time",
    y="month",
    z="shortwave_radiation",
    histfunc="avg",
    nbinsx=24,
    nbinsy=12,
    labels={
        "time": "Time of the Day",
        "month": "Month",
        "shortwave_radiation": "Shortwave Radiation (W/m²)",
    },
    title="Average Hourly Shortwave Radiation by Month",
)
fig.update_yaxes(tickvals=list(range(1, 13)), ticktext=MONTHS)
fig.show()

## Shortwave Radiation by Time of the Day and Season

In [191]:
profile = (
    df.groupby(
        [
            df.index.month.map(SEASONS).rename("season"),
            df.index.strftime("%H:%M").rename("time"),
        ]
    )["shortwave_radiation"]
    .agg(
        P10=lambda s: s.quantile(0.1),
        Median="median",
        P90=lambda s: s.quantile(0.9),
    )
    .reset_index()
)

fig = px.line(
    profile,
    x="time",
    y=["P10", "Median", "P90"],
    facet_col="season",
    facet_col_wrap=2,
    category_orders={"season": ["Winter", "Spring", "Summer", "Autumn"]},
    labels={
        "time": "Time of the Day (UTC)",
        "value": "Shortwave Radiation (W/m²)",
        "variable": "Percentile",
    },
    title="Shortwave Radiation by Time of the Day and Season",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig

## Direct and Diffuse Radiation by Month

In [192]:
monthly_components = (
    df.groupby(df.index.month.rename("month"))[
        ["direct_radiation", "diffuse_radiation"]
    ]
    .mean()
    .rename(columns={"direct_radiation": "Direct", "diffuse_radiation": "Diffuse"})
    .reset_index()
    .melt(id_vars="month", var_name="component", value_name="radiation")
)
monthly_shortwave = df.groupby(df.index.month)["shortwave_radiation"].mean()

fig = px.bar(
    monthly_components,
    x="month",
    y="radiation",
    color="component",
    labels={
        "radiation": "Radiation (W/m²)",
        "month": "Month",
        "component": "Component",
    },
    title="Average Shortwave, Direct and Diffuse Radiation by Month",
)
fig.add_scatter(
    x=monthly_shortwave.index,
    y=monthly_shortwave.values,
    mode="lines+markers",
    name="Shortwave (GHI, Total)",
    line={"color": "black", "width": 2},
    marker={"size": 10},
)
fig.update_xaxes(tickvals=list(range(1, 13)), ticktext=MONTHS)
fig